In [ ]:
# Install required libraries
!pip install -q --upgrade openai chromadb pinecone-client weaviate-client


## Tutorial: Vector Databases — Chroma, Pinecone, Weaviate
Goal: same tiny corpus, same OpenRouter embedding model, three vector stores.
We keep each demo short and block-by-block.


### 1) Setup: API key and tiny corpus


In [ ]:
from getpass import getpass
from openai import OpenAI

OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")
PINECONE_API_KEY = getpass("Enter your Pinecone API key: ")

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

EMBED_MODEL = "openai/text-embedding-3-small"

def embed(text: str):
    r = client.embeddings.create(model=EMBED_MODEL, input=text)
    return r.data[0].embedding

corpus = [
    {"id": "1", "text": "Pinecone is a hosted vector database."},
    {"id": "2", "text": "Chroma is a lightweight local vector store."},
    {"id": "3", "text": "Weaviate is an open-source vector database with hybrid search."},
]
query = "Which tool is lightweight and local?"

print("OpenRouter API key loaded:", bool(OPENROUTER_API_KEY))
print("Pinecone API key loaded:", bool(PINECONE_API_KEY))


### 2) Chroma: create, add, query


Chroma: initialize client and collection


In [ ]:
import chromadb

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="my_collection")

# Generate embeddings through OpenRouter
doc_embeddings = [embed(d["text"]) for d in corpus]
query_embedding = embed(query)

collection.add(
    ids=[d["id"] for d in corpus],
    documents=[d["text"] for d in corpus],
    embeddings=doc_embeddings,
)

# Query Chroma using the OpenRouter-generated query embedding
res = collection.query(query_embeddings=[query_embedding], n_results=2)
res


### 3) Pinecone: create index, upsert, query


In [ ]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

# Embed once using OpenRouter and upsert
vecs = [{
    "id": d["id"],
    "values": embed(d["text"]),
    "metadata": {"text": d["text"]}
} for d in corpus]

INDEX_NAME = "demo-rag-index-sarasai"

index = pc.Index(INDEX_NAME)
up = index.upsert(vectors=vecs, namespace="ns1")

qv = embed(query)
res = index.query(vector=qv, top_k=2, include_metadata=True, namespace="ns1")
res


### 4) Weaviate (optional): quick query


In [ ]:
# Requires a running Weaviate instance or Weaviate Cloud (WCD).
# The embedding vectors should be generated with the OpenRouter `embed()` helper above.
# Example approach:
# import weaviate
# client = weaviate.Client("http://localhost:8080")
# result = client.query.get("Doc", ["text"]).with_near_vector({"vector": embed(query)}).with_limit(2).do()
# result
